In [8]:
"""
Исследовательский анализ данных (EDA)
Набор данных по химическим соединениям — предсказание IC50, CC50, SI
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
DATA_PATH = 'asset-v1_SkillFactory_MIFIML.xlsx'
df = pd.read_excel(DATA_PATH)
df = df.drop(columns=['Unnamed: 0'])

TARGETS = ['IC50, mM', 'CC50, mM', 'SI']
FEATURES = [c for c in df.columns if c not in TARGETS]

print("=" * 60)
print("ОБЗОР НАБОРА ДАННЫХ")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Targets: {TARGETS}")
print(f"Features count: {len(FEATURES)}")
print(f"\nMissing values:\n{df.isna().sum()[df.isna().sum() > 0]}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Базовая статистика
print("\n" + "=" * 60)
print("СТАТИСТИКА ЦЕЛЕВЫХ ПЕРЕМЕННЫХ")
print("=" * 60)
print(df[TARGETS].describe().round(3))

medians = df[TARGETS].median()
print(f"\nMedians:\n{medians.round(4)}")
print(f"\nSI > 8 count: {(df['SI'] > 8).sum()} ({(df['SI'] > 8).mean()*100:.1f}%)")

# Check SI = CC50/IC50 
si_calculated = df['CC50, mM'] / df['IC50, mM']
residuals = (si_calculated - df['SI']).abs()
print(f"\nSI vs CC50/IC50 max diff: {residuals.max():.6f} (SI = CC50/IC50 confirmed)")

# Анализ выбросов (метод МКР)
print("\n" + "=" * 60)
print("АНАЛИЗ ВЫБРОСОВ (МЕТОД МКР)")
print("=" * 60)
for col in TARGETS:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)")

# Асимметрия и эксцесс
print("\n" + "=" * 60)
print("ФОРМА РАСПРЕДЕЛЕНИЯ")
print("=" * 60)
for col in TARGETS:
    sk = stats.skew(df[col])
    ku = stats.kurtosis(df[col])
    _, pval = stats.shapiro(df[col].sample(min(500, len(df)), random_state=42))
    print(f"{col}: skewness={sk:.2f}, kurtosis={ku:.2f}, Shapiro p={pval:.4f}")

# Корреляция признаков с целевыми переменными
corr_ic50 = df[FEATURES].corrwith(df['IC50, mM']).abs().sort_values(ascending=False)
corr_cc50 = df[FEATURES].corrwith(df['CC50, mM']).abs().sort_values(ascending=False)
corr_si   = df[FEATURES].corrwith(df['SI']).abs().sort_values(ascending=False)

print("\n" + "=" * 60)
print("ТОП-10 ПРИЗНАКОВ ПО |КОРРЕЛЯЦИИ| С ЦЕЛЕВЫМИ ПЕРЕМЕННЫМИ")
print("=" * 60)
print(f"\nIC50:\n{corr_ic50.head(10).round(3)}")
print(f"\nCC50:\n{corr_cc50.head(10).round(3)}")
print(f"\nSI:\n{corr_si.head(10).round(3)}")

# Признаки с около-нулевой дисперсией 
nzv = df[FEATURES].std()[df[FEATURES].std() < 0.01]
print(f"\nNear-zero variance features: {len(nzv)}")

# Бинарные / целочисленные признаки
int_feats = [c for c in FEATURES if df[c].nunique() <= 2]
print(f"Binary features: {len(int_feats)}")
print(f"Features with all-zero values: {(df[FEATURES] == 0).all().sum()}")

# Графики 
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('EDA: Распределения целевых переменных', fontsize=16, fontweight='bold')

for i, col in enumerate(TARGETS):
    # Исходное распределение
    ax = axes[i][0]
    ax.hist(df[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} — Raw')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

    # Логарифмическое распределение
    ax = axes[i][1]
    log_vals = np.log1p(df[col])
    ax.hist(log_vals, bins=50, color='darkorange', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} — log1p transform')
    ax.set_xlabel(f'log1p({col})')

    # Boxplot
    ax = axes[i][2]
    ax.boxplot(df[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightgreen', color='green'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col} — Boxplot')
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig('eda_target_distributions.png', dpi=120, bbox_inches='tight')
plt.close()

# Тепловая карта корреляции целевых переменных
fig, ax = plt.subplots(figsize=(6, 5))
corr_matrix = df[TARGETS].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            ax=ax, square=True, linewidths=0.5)
ax.set_title('Корреляционная матрица целевых переменных')
plt.tight_layout()
plt.savefig('eda_target_corr.png', dpi=120, bbox_inches='tight')
plt.close()

# Топ корреляций признаков с целевыми переменными
top_feats = list(set(corr_ic50.head(15).index) | set(corr_cc50.head(15).index))
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
corr_sub = df[top_feats + TARGETS].corr()
sns.heatmap(corr_sub[TARGETS].drop(TARGETS).sort_values('IC50, mM', key=abs, ascending=False).head(20),
            annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[0], linewidths=0.3)
axes[0].set_title('Топ корреляций признаков с целевыми переменными')

# Диаграмма рассеяния целевых переменных
axes[1].scatter(df['IC50, mM'], df['SI'], alpha=0.3, s=10, color='steelblue')
axes[1].set_xlabel('IC50, mM')
axes[1].set_ylabel('SI')
axes[1].set_title('IC50 vs SI (исходные значения)')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('eda_feature_corr.png', dpi=120, bbox_inches='tight')
plt.close()

# Распределение меток классификации 
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
labels_info = [
    ('IC50 > median', df['IC50, mM'] > df['IC50, mM'].median()),
    ('CC50 > median', df['CC50, mM'] > df['CC50, mM'].median()),
    ('SI > median',   df['SI'] > df['SI'].median()),
    ('SI > 8',        df['SI'] > 8),
]
for ax, (name, mask) in zip(axes, labels_info):
    counts = mask.value_counts()
    ax.bar(['False', 'True'], [counts.get(False, 0), counts.get(True, 0)],
           color=['#e74c3c', '#2ecc71'])
    ax.set_title(name)
    ax.set_ylabel('Count')
    for j, v in enumerate([counts.get(False, 0), counts.get(True, 0)]):
        ax.text(j, v + 5, str(v), ha='center', fontweight='bold')

plt.suptitle('Распределение меток классификации', fontsize=13)
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=120, bbox_inches='tight')
plt.close()

print("\n EDA завершён. Графики сохранены.")


ОБЗОР НАБОРА ДАННЫХ
Shape: (1001, 213)
Targets: ['IC50, mM', 'CC50, mM', 'SI']
Features count: 210

Missing values:
MaxPartialCharge       3
MinPartialCharge       3
MaxAbsPartialCharge    3
MinAbsPartialCharge    3
BCUT2D_MWHI            3
BCUT2D_MWLOW           3
BCUT2D_CHGHI           3
BCUT2D_CHGLO           3
BCUT2D_LOGPHI          3
BCUT2D_LOGPLOW         3
BCUT2D_MRHI            3
BCUT2D_MRLOW           3
dtype: int64

Duplicate rows: 32

СТАТИСТИКА ЦЕЛЕВЫХ ПЕРЕМЕННЫХ
       IC50, mM  CC50, mM         SI
count  1001.000  1001.000   1001.000
mean    222.805   589.111     72.509
std     402.170   642.868    684.483
min       0.004     0.701      0.011
25%      12.515    99.999      1.433
50%      46.585   411.039      3.846
75%     224.976   894.089     16.567
max    4128.529  4538.976  15620.600

Medians:
IC50, mM     46.5852
CC50, mM    411.0393
SI            3.8462
dtype: float64

SI > 8 count: 357 (35.7%)

SI vs CC50/IC50 max diff: 0.000000 (SI = CC50/IC50 confirmed)

АНАЛИЗ В

In [9]:
"""
Задача 1: Регрессия для IC50
Модели: LinearRegression, Ridge, Lasso, RandomForest, GradientBoosting,
        XGBoost, LightGBM — с GridSearchCV / RandomizedSearchCV
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from utils import load_data, prepare_regression, print_regression_metrics

TARGET = 'IC50, mM'

print("=" * 60)
print(f"REGRESSION: {TARGET}")
print("=" * 60)

df = load_data()
X_train, X_test, y_train, y_test = prepare_regression(df, TARGET, log_transform=True)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target (log1p) — mean: {y_train.mean():.3f}, std: {y_train.std():.3f}\n")

results = []

# 1. Линейная регрессия (базовая модель)
lr = LinearRegression()
lr.fit(X_train, y_train)
r = print_regression_metrics("LinearRegression (baseline)", y_test, lr.predict(X_test))
results.append(r)

# 2. Гребневая регрессия
ridge_gs = GridSearchCV(
    Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
ridge_gs.fit(X_train, y_train)
print(f"  Ridge best alpha: {ridge_gs.best_params_}")
r = print_regression_metrics("Ridge (best alpha)", y_test, ridge_gs.predict(X_test))
results.append(r)

# 3. Лассо 
lasso_gs = GridSearchCV(
    Lasso(max_iter=5000), {'alpha': [0.001, 0.01, 0.1, 1]},
    cv=5, scoring='r2', n_jobs=-1
)
lasso_gs.fit(X_train, y_train)
print(f"  Lasso best alpha: {lasso_gs.best_params_}")
r = print_regression_metrics("Lasso (best alpha)", y_test, lasso_gs.predict(X_test))
results.append(r)

# 4. Random Forest
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 300], 'max_depth': [None, 10, 20], 'min_samples_leaf': [1, 3]},
    cv=5, scoring='r2', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
print(f"  RF best params: {rf_gs.best_params_}")
r = print_regression_metrics("RandomForest (tuned)", y_test, rf_gs.predict(X_test))
results.append(r)

# 5. Градиентный спуск
gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=5, scoring='r2', n_jobs=-1
)
gb_gs.fit(X_train, y_train)
print(f"  GB best params: {gb_gs.best_params_}")
r = print_regression_metrics("GradientBoosting (tuned)", y_test, gb_gs.predict(X_test))
results.append(r)

# 6. XGBoost 
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
    cv=5, scoring='r2', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best params: {xgb_gs.best_params_}")
r = print_regression_metrics("XGBoost (tuned)", y_test, xgb_gs.predict(X_test))
results.append(r)

# 7. LightGBM 
lgb_gs = GridSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
    cv=5, scoring='r2', n_jobs=-1
)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best params: {lgb_gs.best_params_}")
r = print_regression_metrics("LightGBM (tuned)", y_test, lgb_gs.predict(X_test))
results.append(r)

# Сводная таблица результатов 
results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — R²={best['R2']:.4f}")

# Важность признаков лучшей модели 
best_model_obj = lgb_gs if 'LightGBM' in best['model'] else (
    xgb_gs if 'XGBoost' in best['model'] else rf_gs)
try:
    fi = pd.Series(
        best_model_obj.best_estimator_.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(9, 6))
    fi.sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'Топ-20 важных признаков — Регрессия IC50 ({best["model"]})')
    ax.set_xlabel('Важность')
    plt.tight_layout()
    plt.savefig('reg_ic50_importance.png', dpi=120)
    plt.close()
except Exception:
    pass

# Фактические vs предсказанные значения 
best_preds = best_model_obj.predict(X_test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_preds, alpha=0.4, s=15)
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--')
ax.set_xlabel('Факт (log1p IC50)')
ax.set_ylabel('Предсказание')
ax.set_title(f'Регрессия IC50 — Факт vs Прогноз ({best["model"]})')
plt.tight_layout()
plt.savefig('reg_ic50_pred.png', dpi=120)
plt.close()

results_df.to_csv('reg_ic50_results.csv', index=False)
print("\n Регрессия IC50 завершена.")


REGRESSION: IC50, mM
Train: (800, 192), Test: (201, 192)
Target (log1p) — mean: 3.941, std: 1.833

  LinearRegression (baseline)         R²=0.3306  MAE=1.269  RMSE=1.605
  Ridge best alpha: {'alpha': 10}
  Ridge (best alpha)                  R²=0.3197  MAE=1.286  RMSE=1.618
  Lasso best alpha: {'alpha': 1}
  Lasso (best alpha)                  R²=0.1137  MAE=1.525  RMSE=1.847
  RF best params: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 300}
  RandomForest (tuned)                R²=0.4558  MAE=1.143  RMSE=1.447
  GB best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
  GradientBoosting (tuned)            R²=0.4267  MAE=1.188  RMSE=1.486
  XGB best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  XGBoost (tuned)                     R²=0.4510  MAE=1.176  RMSE=1.454
  LGB best params: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    R²=0.4336  MAE=1.150  RMSE=1.477

СВОДНАЯ ТАБЛИЦА

In [10]:
"""
Задача 2: Регрессия для CC50
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from utils import load_data, prepare_regression, print_regression_metrics

TARGET = 'CC50, mM'

print("=" * 60)
print(f"REGRESSION: {TARGET}")
print("=" * 60)

df = load_data()
X_train, X_test, y_train, y_test = prepare_regression(df, TARGET, log_transform=True)
print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

results = []

lr = LinearRegression()
lr.fit(X_train, y_train)
results.append(print_regression_metrics("LinearRegression (baseline)", y_test, lr.predict(X_test)))

ridge_gs = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='r2', n_jobs=-1)
ridge_gs.fit(X_train, y_train)
print(f"  Ridge best: {ridge_gs.best_params_}")
results.append(print_regression_metrics("Ridge (tuned)", y_test, ridge_gs.predict(X_test)))

lasso_gs = GridSearchCV(Lasso(max_iter=5000), {'alpha': [0.001, 0.01, 0.1, 1]}, cv=5, scoring='r2', n_jobs=-1)
lasso_gs.fit(X_train, y_train)
print(f"  Lasso best: {lasso_gs.best_params_}")
results.append(print_regression_metrics("Lasso (tuned)", y_test, lasso_gs.predict(X_test)))

rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 300], 'max_depth': [None, 10, 20], 'min_samples_leaf': [1, 3]},
    cv=5, scoring='r2', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
results.append(print_regression_metrics("RandomForest (tuned)", y_test, rf_gs.predict(X_test)))

gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=5, scoring='r2', n_jobs=-1
)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
results.append(print_regression_metrics("GradientBoosting (tuned)", y_test, gb_gs.predict(X_test)))

xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
    cv=5, scoring='r2', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
results.append(print_regression_metrics("XGBoost (tuned)", y_test, xgb_gs.predict(X_test)))

lgb_gs = GridSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
    cv=5, scoring='r2', n_jobs=-1
)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
results.append(print_regression_metrics("LightGBM (tuned)", y_test, lgb_gs.predict(X_test)))

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — R²={best['R2']:.4f}")

# Графики лучшей модели
best_obj = lgb_gs if 'LightGBM' in best['model'] else (xgb_gs if 'XGBoost' in best['model'] else rf_gs)
try:
    fi = pd.Series(best_obj.best_estimator_.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    fi.sort_values().plot.barh(ax=ax, color='darkorange')
    ax.set_title(f'Топ-20 важных признаков — Регрессия CC50 ({best["model"]})')
    plt.tight_layout()
    plt.savefig('reg_cc50_importance.png', dpi=120)
    plt.close()
except Exception:
    pass

best_preds = best_obj.predict(X_test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_preds, alpha=0.4, s=15, color='darkorange')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--')
ax.set_xlabel('Факт (log1p CC50)')
ax.set_ylabel('Предсказание')
ax.set_title(f'Регрессия CC50 — Факт vs Прогноз ({best["model"]})')
plt.tight_layout()
plt.savefig('reg_cc50_pred.png', dpi=120)
plt.close()

results_df.to_csv('reg_cc50_results.csv', index=False)
print("\n Регрессия CC50 завершена.")


REGRESSION: CC50, mM
Train: (800, 192), Test: (201, 192)

  LinearRegression (baseline)         R²=0.3602  MAE=0.930  RMSE=1.206
  Ridge best: {'alpha': 10}
  Ridge (tuned)                       R²=0.3678  MAE=0.929  RMSE=1.199
  Lasso best: {'alpha': 1}
  Lasso (tuned)                       R²=0.1841  MAE=1.148  RMSE=1.362
  RF best: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 100}
  RandomForest (tuned)                R²=0.4289  MAE=0.821  RMSE=1.140
  GB best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  GradientBoosting (tuned)            R²=0.4232  MAE=0.822  RMSE=1.145
  XGB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
  XGBoost (tuned)                     R²=0.4004  MAE=0.828  RMSE=1.168
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    R²=0.4193  MAE=0.799  RMSE=1.149

СВОДНАЯ ТАБЛИЦА
                      model       R2      MAE     RMSE
       RandomForest (tuned

In [11]:
"""
Задача 3: Регрессия для SI
Примечание: SI = CC50 / IC50. Модель строится только по молекулярным признакам.
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from utils import load_data, prepare_regression, print_regression_metrics

TARGET = 'SI'

print("=" * 60)
print(f"REGRESSION: {TARGET}")
print("=" * 60)

df = load_data()
X_train, X_test, y_train, y_test = prepare_regression(df, TARGET, log_transform=True)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"SI range: {df[TARGET].min():.3f} – {df[TARGET].max():.1f}, median={df[TARGET].median():.3f}\n")

results = []

lr = LinearRegression()
lr.fit(X_train, y_train)
results.append(print_regression_metrics("LinearRegression (baseline)", y_test, lr.predict(X_test)))

ridge_gs = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='r2', n_jobs=-1)
ridge_gs.fit(X_train, y_train)
print(f"  Ridge best: {ridge_gs.best_params_}")
results.append(print_regression_metrics("Ridge (tuned)", y_test, ridge_gs.predict(X_test)))

lasso_gs = GridSearchCV(Lasso(max_iter=5000), {'alpha': [0.001, 0.01, 0.1, 1]}, cv=5, scoring='r2', n_jobs=-1)
lasso_gs.fit(X_train, y_train)
print(f"  Lasso best: {lasso_gs.best_params_}")
results.append(print_regression_metrics("Lasso (tuned)", y_test, lasso_gs.predict(X_test)))

rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 300], 'max_depth': [None, 10, 20], 'min_samples_leaf': [1, 3]},
    cv=5, scoring='r2', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
results.append(print_regression_metrics("RandomForest (tuned)", y_test, rf_gs.predict(X_test)))

gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=5, scoring='r2', n_jobs=-1
)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
results.append(print_regression_metrics("GradientBoosting (tuned)", y_test, gb_gs.predict(X_test)))

xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
    cv=5, scoring='r2', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
results.append(print_regression_metrics("XGBoost (tuned)", y_test, xgb_gs.predict(X_test)))

lgb_gs = GridSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
    cv=5, scoring='r2', n_jobs=-1
)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
results.append(print_regression_metrics("LightGBM (tuned)", y_test, lgb_gs.predict(X_test)))

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — R²={best['R2']:.4f}")

best_obj = lgb_gs if 'LightGBM' in best['model'] else (xgb_gs if 'XGBoost' in best['model'] else rf_gs)
try:
    fi = pd.Series(best_obj.best_estimator_.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    fi.sort_values().plot.barh(ax=ax, color='green')
    ax.set_title(f'Топ-20 важных признаков — Регрессия SI ({best["model"]})')
    plt.tight_layout()
    plt.savefig('reg_si_importance.png', dpi=120)
    plt.close()
except Exception:
    pass

best_preds = best_obj.predict(X_test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_preds, alpha=0.4, s=15, color='green')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--')
ax.set_xlabel('Факт (log1p SI)')
ax.set_ylabel('Предсказание')
ax.set_title(f'Регрессия SI — Факт vs Прогноз ({best["model"]})')
plt.tight_layout()
plt.savefig('reg_si_pred.png', dpi=120)
plt.close()

results_df.to_csv('reg_si_results.csv', index=False)
print("\n Регрессия SI завершена.")


REGRESSION: SI
Train: (800, 192), Test: (201, 192)
SI range: 0.011 – 15620.6, median=3.846

  LinearRegression (baseline)         R²=0.1096  MAE=1.081  RMSE=1.468
  Ridge best: {'alpha': 100}
  Ridge (tuned)                       R²=0.1560  MAE=1.053  RMSE=1.429
  Lasso best: {'alpha': 0.1}
  Lasso (tuned)                       R²=0.1152  MAE=1.096  RMSE=1.463
  RF best: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 300}
  RandomForest (tuned)                R²=0.3388  MAE=0.931  RMSE=1.265
  GB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  GradientBoosting (tuned)            R²=0.2965  MAE=0.982  RMSE=1.305
  XGB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  XGBoost (tuned)                     R²=0.3007  MAE=0.968  RMSE=1.301
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    R²=0.3231  MAE=0.896  RMSE=1.280

СВОДНАЯ ТАБЛИЦА
                      model       R2      M

In [12]:
"""
Задача 4: Классификация — IC50 > медианы
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, RocCurveDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utils import load_data, prepare_classification, print_classification_metrics

TARGET = 'IC50, mM'

print("=" * 60)
print(f"CLASSIFICATION: {TARGET} > median")
print("=" * 60)

df = load_data()
median_val = df[TARGET].median()
print(f"Median IC50: {median_val:.4f}")
print(f"Class balance: {(df[TARGET] > median_val).mean()*100:.1f}% positive\n")

X_train, X_test, y_train, y_test = prepare_classification(df, TARGET, use_median=True)

results = []

# 1. Логистическая регрессия
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.01, 0.1, 1, 10]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
lr_gs.fit(X_train, y_train)
print(f"  LR best: {lr_gs.best_params_}")
p = lr_gs.predict(X_test)
pb = lr_gs.predict_proba(X_test)[:, 1]
results.append(print_classification_metrics("LogisticRegression (tuned)", y_test, p, pb))

# 2. Random Forest 
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [100, 300], 'max_depth': [None, 10], 'min_samples_leaf': [1, 3]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
p = rf_gs.predict(X_test)
pb = rf_gs.predict_proba(X_test)[:, 1]
results.append(print_classification_metrics("RandomForest (tuned)", y_test, p, pb))

# 3. Градиентный спуск 
gb_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
p = gb_gs.predict(X_test)
pb = gb_gs.predict_proba(X_test)[:, 1]
results.append(print_classification_metrics("GradientBoosting (tuned)", y_test, p, pb))

# 4. XGBoost 
xgb_gs = GridSearchCV(
    XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss'),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
p = xgb_gs.predict(X_test)
pb = xgb_gs.predict_proba(X_test)[:, 1]
results.append(print_classification_metrics("XGBoost (tuned)", y_test, p, pb))

# 5. LightGBM 
lgb_gs = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
p = lgb_gs.predict(X_test)
pb = lgb_gs.predict_proba(X_test)[:, 1]
results.append(print_classification_metrics("LightGBM (tuned)", y_test, p, pb))

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — AUC={best['AUC']:.4f}")

# ROC-кривые
fig, ax = plt.subplots(figsize=(7, 6))
models_map = {
    'LogisticRegression (tuned)': lr_gs,
    'RandomForest (tuned)': rf_gs,
    'GradientBoosting (tuned)': gb_gs,
    'XGBoost (tuned)': xgb_gs,
    'LightGBM (tuned)': lgb_gs,
}
for name, model in models_map.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name, alpha=0.7)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC-кривые — IC50 > медианы')
plt.tight_layout()
plt.savefig('cls_ic50_roc.png', dpi=120)
plt.close()

# Отчёт классификации лучшей модели
best_obj = models_map.get(best['model'], lgb_gs)
print("\nОтчёт классификации лучшей модели:")
print(classification_report(y_test, best_obj.predict(X_test)))

results_df.to_csv('cls_ic50_results.csv', index=False)
print("\n Классификация IC50 завершена.")


CLASSIFICATION: IC50, mM > median
Median IC50: 46.5852
Class balance: 50.0% positive

  LR best: {'C': 0.01}
  LogisticRegression (tuned)          Acc=0.4975  F1=0.6645  AUC=0.4626
  RF best: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 300}
  RandomForest (tuned)                Acc=0.7363  F1=0.7512  AUC=0.7947
  GB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  GradientBoosting (tuned)            Acc=0.7114  F1=0.7339  AUC=0.7720
  XGB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  XGBoost (tuned)                     Acc=0.7214  F1=0.7431  AUC=0.7829
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    Acc=0.6866  F1=0.7070  AUC=0.7627

СВОДНАЯ ТАБЛИЦА
                     model  Accuracy       F1      AUC
      RandomForest (tuned)  0.736318 0.751174 0.794703
           XGBoost (tuned)  0.721393 0.743119 0.782921
  GradientBoosting (tuned)  0.711443 0.733945 0.771980
 

In [13]:
"""
Задача 5: Классификация — CC50 > медианы
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, RocCurveDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utils import load_data, prepare_classification, print_classification_metrics

TARGET = 'CC50, mM'

print("=" * 60)
print(f"CLASSIFICATION: {TARGET} > median")
print("=" * 60)

df = load_data()
median_val = df[TARGET].median()
print(f"Median CC50: {median_val:.4f}")
print(f"Class balance: {(df[TARGET] > median_val).mean()*100:.1f}% positive\n")

X_train, X_test, y_train, y_test = prepare_classification(df, TARGET, use_median=True)

results = []

lr_gs = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                     {'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='roc_auc', n_jobs=-1)
lr_gs.fit(X_train, y_train)
print(f"  LR best: {lr_gs.best_params_}")
results.append(print_classification_metrics("LogisticRegression (tuned)", y_test,
                                             lr_gs.predict(X_test), lr_gs.predict_proba(X_test)[:, 1]))

rf_gs = GridSearchCV(RandomForestClassifier(random_state=42),
                     {'n_estimators': [100, 300], 'max_depth': [None, 10], 'min_samples_leaf': [1, 3]},
                     cv=5, scoring='roc_auc', n_jobs=-1)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
results.append(print_classification_metrics("RandomForest (tuned)", y_test,
                                             rf_gs.predict(X_test), rf_gs.predict_proba(X_test)[:, 1]))

gb_gs = GridSearchCV(GradientBoostingClassifier(random_state=42),
                     {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
                     cv=5, scoring='roc_auc', n_jobs=-1)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
results.append(print_classification_metrics("GradientBoosting (tuned)", y_test,
                                             gb_gs.predict(X_test), gb_gs.predict_proba(X_test)[:, 1]))

xgb_gs = GridSearchCV(XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss'),
                      {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
                      cv=5, scoring='roc_auc', n_jobs=-1)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
results.append(print_classification_metrics("XGBoost (tuned)", y_test,
                                             xgb_gs.predict(X_test), xgb_gs.predict_proba(X_test)[:, 1]))

lgb_gs = GridSearchCV(LGBMClassifier(random_state=42, verbose=-1),
                      {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
                      cv=5, scoring='roc_auc', n_jobs=-1)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
results.append(print_classification_metrics("LightGBM (tuned)", y_test,
                                             lgb_gs.predict(X_test), lgb_gs.predict_proba(X_test)[:, 1]))

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n🏆 Лучшая модель: {best['model']} — AUC={best['AUC']:.4f}")

models_map = {
    'LogisticRegression (tuned)': lr_gs, 'RandomForest (tuned)': rf_gs,
    'GradientBoosting (tuned)': gb_gs, 'XGBoost (tuned)': xgb_gs,
    'LightGBM (tuned)': lgb_gs,
}
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models_map.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name, alpha=0.7)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC-кривые — CC50 > медианы')
plt.tight_layout()
plt.savefig('cls_cc50_roc.png', dpi=120)
plt.close()

best_obj = models_map.get(best['model'], lgb_gs)
print("\nОтчёт классификации лучшей модели:")
print(classification_report(y_test, best_obj.predict(X_test)))

results_df.to_csv('cls_cc50_results.csv', index=False)
print("\n Классификация CC50 завершена.")


CLASSIFICATION: CC50, mM > median
Median CC50: 411.0393
Class balance: 49.9% positive

  LR best: {'C': 0.01}
  LogisticRegression (tuned)          Acc=0.5025  F1=0.0000  AUC=0.5312
  RF best: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 300}
  RandomForest (tuned)                Acc=0.6965  F1=0.7189  AUC=0.8337
  GB best: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}
  GradientBoosting (tuned)            Acc=0.7114  F1=0.7264  AUC=0.8362
  XGB best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  XGBoost (tuned)                     Acc=0.7164  F1=0.7299  AUC=0.8366
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    Acc=0.7313  F1=0.7477  AUC=0.8562

СВОДНАЯ ТАБЛИЦА
                     model  Accuracy       F1      AUC
          LightGBM (tuned)  0.731343 0.747664 0.856238
           XGBoost (tuned)  0.716418 0.729858 0.836634
  GradientBoosting (tuned)  0.711443 0.726415 0.836238
 

In [14]:
"""
Задача 6: Классификация — SI > медианы
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, RocCurveDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utils import load_data, prepare_classification, print_classification_metrics

TARGET = 'SI'

print("=" * 60)
print(f"CLASSIFICATION: {TARGET} > median")
print("=" * 60)

df = load_data()
median_val = df[TARGET].median()
print(f"Median SI: {median_val:.4f}")
print(f"Class balance: {(df[TARGET] > median_val).mean()*100:.1f}% positive\n")

X_train, X_test, y_train, y_test = prepare_classification(df, TARGET, use_median=True)

results = []

lr_gs = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                     {'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='roc_auc', n_jobs=-1)
lr_gs.fit(X_train, y_train)
print(f"  LR best: {lr_gs.best_params_}")
results.append(print_classification_metrics("LogisticRegression (tuned)", y_test,
                                             lr_gs.predict(X_test), lr_gs.predict_proba(X_test)[:, 1]))

rf_gs = GridSearchCV(RandomForestClassifier(random_state=42),
                     {'n_estimators': [100, 300], 'max_depth': [None, 10], 'min_samples_leaf': [1, 3]},
                     cv=5, scoring='roc_auc', n_jobs=-1)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
results.append(print_classification_metrics("RandomForest (tuned)", y_test,
                                             rf_gs.predict(X_test), rf_gs.predict_proba(X_test)[:, 1]))

gb_gs = GridSearchCV(GradientBoostingClassifier(random_state=42),
                     {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
                     cv=5, scoring='roc_auc', n_jobs=-1)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
results.append(print_classification_metrics("GradientBoosting (tuned)", y_test,
                                             gb_gs.predict(X_test), gb_gs.predict_proba(X_test)[:, 1]))

xgb_gs = GridSearchCV(XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss'),
                      {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
                      cv=5, scoring='roc_auc', n_jobs=-1)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
results.append(print_classification_metrics("XGBoost (tuned)", y_test,
                                             xgb_gs.predict(X_test), xgb_gs.predict_proba(X_test)[:, 1]))

lgb_gs = GridSearchCV(LGBMClassifier(random_state=42, verbose=-1),
                      {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
                      cv=5, scoring='roc_auc', n_jobs=-1)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
results.append(print_classification_metrics("LightGBM (tuned)", y_test,
                                             lgb_gs.predict(X_test), lgb_gs.predict_proba(X_test)[:, 1]))

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — AUC={best['AUC']:.4f}")

models_map = {
    'LogisticRegression (tuned)': lr_gs, 'RandomForest (tuned)': rf_gs,
    'GradientBoosting (tuned)': gb_gs, 'XGBoost (tuned)': xgb_gs,
    'LightGBM (tuned)': lgb_gs,
}
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models_map.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name, alpha=0.7)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC-кривые — SI > медианы')
plt.tight_layout()
plt.savefig('cls_si_median_roc.png', dpi=120)
plt.close()

best_obj = models_map.get(best['model'], lgb_gs)
print("\nОтчёт классификации лучшей модели:")
print(classification_report(y_test, best_obj.predict(X_test)))

results_df.to_csv('cls_si_median_results.csv', index=False)
print("\n Классификация SI > медианы завершена.")


CLASSIFICATION: SI > median
Median SI: 3.8462
Class balance: 50.0% positive

  LR best: {'C': 0.01}
  LogisticRegression (tuned)          Acc=0.5025  F1=0.0000  AUC=0.5560
  RF best: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 300}
  RandomForest (tuned)                Acc=0.6567  F1=0.6188  AUC=0.6878
  GB best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  GradientBoosting (tuned)            Acc=0.6567  F1=0.6425  AUC=0.6929
  XGB best: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
  XGBoost (tuned)                     Acc=0.6617  F1=0.6383  AUC=0.6887
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (tuned)                    Acc=0.6418  F1=0.6170  AUC=0.6704

СВОДНАЯ ТАБЛИЦА
                     model  Accuracy       F1      AUC
  GradientBoosting (tuned)  0.656716 0.642487 0.692921
           XGBoost (tuned)  0.661692 0.638298 0.688663
      RandomForest (tuned)  0.656716 0.618785 0.687772
          Li

In [15]:
"""
Задача 7: Классификация — SI > 8
SI > 8 — распространённый фармакологический порог, означающий, что соединение
в 8 раз токсичнее для раковых/вирусных клеток, чем для здоровых — значимый рубеж.
"""

#import sys, os
#sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, RocCurveDisplay, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from utils import load_data, prepare_classification, print_classification_metrics

TARGET = 'SI'
THRESHOLD = 8

print("=" * 60)
print(f"CLASSIFICATION: {TARGET} > {THRESHOLD}")
print("=" * 60)

df = load_data()
pos_frac = (df[TARGET] > THRESHOLD).mean()
print(f"SI > 8: {pos_frac*100:.1f}% positive ({(df[TARGET] > THRESHOLD).sum()} / {len(df)})")
print("Биологический смысл: SI > 8 указывает на перспективное терапевтическое окно\n")

X_train, X_test, y_train, y_test = prepare_classification(df, TARGET, threshold=THRESHOLD)

# Веса классов для учёта дисбаланса
scale = int((y_train == 0).sum() / (y_train == 1).sum())

results = []

lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    {'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='roc_auc', n_jobs=-1
)
lr_gs.fit(X_train, y_train)
print(f"  LR best: {lr_gs.best_params_}")
results.append(print_classification_metrics("LogisticRegression (balanced)", y_test,
                                             lr_gs.predict(X_test), lr_gs.predict_proba(X_test)[:, 1]))

rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    {'n_estimators': [100, 300], 'max_depth': [None, 10], 'min_samples_leaf': [1, 3]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
print(f"  RF best: {rf_gs.best_params_}")
results.append(print_classification_metrics("RandomForest (balanced)", y_test,
                                             rf_gs.predict(X_test), rf_gs.predict_proba(X_test)[:, 1]))

gb_gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
gb_gs.fit(X_train, y_train)
print(f"  GB best: {gb_gs.best_params_}")
results.append(print_classification_metrics("GradientBoosting (tuned)", y_test,
                                             gb_gs.predict(X_test), gb_gs.predict_proba(X_test)[:, 1]))

xgb_gs = GridSearchCV(
    XGBClassifier(random_state=42, verbosity=0, eval_metric='logloss',
                  scale_pos_weight=scale),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 6]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
print(f"  XGB best: {xgb_gs.best_params_}")
results.append(print_classification_metrics("XGBoost (scale_pos_weight)", y_test,
                                             xgb_gs.predict(X_test), xgb_gs.predict_proba(X_test)[:, 1]))

lgb_gs = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1], 'num_leaves': [31, 63]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
lgb_gs.fit(X_train, y_train)
print(f"  LGB best: {lgb_gs.best_params_}")
results.append(print_classification_metrics("LightGBM (balanced)", y_test,
                                             lgb_gs.predict(X_test), lgb_gs.predict_proba(X_test)[:, 1]))

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)
print(results_df.to_string(index=False))
best = results_df.iloc[0]
print(f"\n Лучшая модель: {best['model']} — AUC={best['AUC']:.4f}")

models_map = {
    'LogisticRegression (balanced)': lr_gs, 'RandomForest (balanced)': rf_gs,
    'GradientBoosting (tuned)': gb_gs, 'XGBoost (scale_pos_weight)': xgb_gs,
    'LightGBM (balanced)': lgb_gs,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
for name, model in models_map.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name, alpha=0.7)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title('ROC-кривые — SI > 8')

best_obj = models_map.get(best['model'], lgb_gs)
cm = confusion_matrix(y_test, best_obj.predict(X_test))
disp = ConfusionMatrixDisplay(cm, display_labels=['SI≤8', 'SI>8'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Матрица ошибок — {best["model"]}')

plt.tight_layout()
plt.savefig('cls_si8_roc_cm.png', dpi=120)
plt.close()

print("\nОтчёт классификации лучшей модели:")
print(classification_report(y_test, best_obj.predict(X_test), target_names=['SI≤8', 'SI>8']))

results_df.to_csv('cls_si8_results.csv', index=False)
print("\n Классификация SI > 8 завершена.")


CLASSIFICATION: SI > 8
SI > 8: 35.7% positive (357 / 1001)
Биологический смысл: SI > 8 указывает на перспективное терапевтическое окно

  LR best: {'C': 0.01}
  LogisticRegression (balanced)       Acc=0.6418  F1=0.0000  AUC=0.6347
  RF best: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 100}
  RandomForest (balanced)             Acc=0.6866  F1=0.5401  AUC=0.7421
  GB best: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
  GradientBoosting (tuned)            Acc=0.6965  F1=0.5547  AUC=0.7001
  XGB best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
  XGBoost (scale_pos_weight)          Acc=0.7164  F1=0.5714  AUC=0.7259
  LGB best: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
  LightGBM (balanced)                 Acc=0.6915  F1=0.5634  AUC=0.7244

СВОДНАЯ ТАБЛИЦА
                        model  Accuracy       F1      AUC
      RandomForest (balanced)  0.686567 0.540146 0.742087
   XGBoost (scale_pos_weight)  0.716418 0.571429 0.72593